# Step 2: Parquet and Partitioning

This notebook converts the raw CSV file into a format a query engine can exploit efficiently: **Parquet**, a columnar, compressed file format with an embedded schema. DuckDB performs the conversion directly, without requiring a separate ETL framework.

The output is also **partitioned** by year and month. A partition is simply a folder: `lake/verkauf/jahr=2026/monat=08/data.parquet` communicates to any engine that everything within it belongs to August 2026, purely through the folder name — no database or catalog is required.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("SELECT * FROM read_csv_auto('lake/raw/sales.csv') LIMIT 5").show()

## Converting to Partitioned Parquet

The statement `COPY ... TO ... (FORMAT PARQUET, PARTITION_BY (...))` reads the CSV, derives `jahr` (year) and `monat` (month) from the date column, and writes one Parquet file per partition folder — all within a single SQL statement.

In [ ]:
con.sql("""
    COPY (
        SELECT
            *,
            EXTRACT(year FROM date)  AS jahr,
            EXTRACT(month FROM date) AS monat
        FROM read_csv_auto('lake/raw/sales.csv')
    ) TO 'lake/verkauf' (
        FORMAT PARQUET,
        PARTITION_BY (jahr, monat),
        OVERWRITE_OR_IGNORE 1
    )
""")
print("Done.")

## Inspecting the Partition Layout

The result is a standard folder tree, viewable in the file explorer or listed directly below.

In [ ]:
for path in sorted(Path("lake/verkauf").rglob("*.parquet"))[:12]:
    print(path)

## Comparing File Size: CSV vs. Parquet

A reduction of approximately 5–10x is expected: the columnar layout compresses repeated values (region, product) considerably more effectively than row-based text.

In [ ]:
def dir_size_mb(path: str) -> float:
    total_bytes = sum(
        f.stat().st_size for f in Path(path).rglob("*") if f.is_file()
    )
    return total_bytes / (1024 * 1024)

csv_mb = dir_size_mb("lake/raw")
parquet_mb = dir_size_mb("lake/verkauf")

print(f"CSV (raw):          {csv_mb:8.1f} MB")
print(f"Parquet (curated):  {parquet_mb:8.1f} MB")
print(f"Reduction factor:   {csv_mb / parquet_mb:8.1f}x")

Two changes occurred, both observable without querying a database:

1. **Compression** — the on-disk footprint was reduced by roughly 5–10x.
2. **Partitioning** — the data is now organized into `jahr=`/`monat=` folders, which the query engine in the next notebook can use to skip irrelevant work entirely.